## Part 1: Preprocessing

In [1]:
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras import layers
from sklearn.preprocessing import OneHotEncoder

#  Import and read the attrition data
attrition_df = pd.read_csv('https://static.bc-edx.com/ai/ail-v-1-0/m19/lms/datasets/attrition.csv')
attrition_df.head()

I0000 00:00:1792530815.664285    1509 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792530815.665124    1509 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1792530817.973150    1509 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792530817.973814    1509 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,HourlyRate,JobInvolvement,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,Sales,1,2,Life Sciences,2,94,3,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,Research & Development,8,1,Life Sciences,3,61,2,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,Research & Development,2,2,Other,4,92,2,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,Research & Development,3,4,Life Sciences,4,56,3,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,Research & Development,2,1,Medical,1,40,3,...,3,4,1,6,3,3,2,2,2,2


In [2]:
# Determine the number of unique values in each column.
attrition_df.nunique()

Age                         43
Attrition                    2
BusinessTravel               3
Department                   3
DistanceFromHome            29
Education                    5
EducationField               6
EnvironmentSatisfaction      4
HourlyRate                  71
JobInvolvement               4
JobLevel                     5
JobRole                      9
JobSatisfaction              4
MaritalStatus                3
NumCompaniesWorked          10
OverTime                     2
PercentSalaryHike           15
PerformanceRating            2
RelationshipSatisfaction     4
StockOptionLevel             4
TotalWorkingYears           40
TrainingTimesLastYear        7
WorkLifeBalance              4
YearsAtCompany              37
YearsInCurrentRole          19
YearsSinceLastPromotion     16
YearsWithCurrManager        18
dtype: int64

In [3]:
# Create y_df with the Attrition and Department columns
y_df = attrition_df[['Attrition', 'Department']]


In [4]:
# Create a list of at least 10 column names to use as X data
selected_columns = [
    'Age', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction',
    'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'NumCompaniesWorked',
    'PercentSalaryHike', 'TotalWorkingYears', 'WorkLifeBalance'
]

# Create X_df using your selected columns
X_df = attrition_df[selected_columns]

# Show the data types for X_df
X_df.dtypes


Age                        int64
DistanceFromHome           int64
Education                  int64
EnvironmentSatisfaction    int64
JobInvolvement             int64
JobLevel                   int64
JobSatisfaction            int64
NumCompaniesWorked         int64
PercentSalaryHike          int64
TotalWorkingYears          int64
WorkLifeBalance            int64
dtype: object

In [5]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_df, y_df)

In [6]:
# Convert your X data to numeric data types however you see fit
# Add new code cells as necessary
X_df = X_df.apply(pd.to_numeric, errors='coerce')

In [7]:
# Create a StandardScaler
scaler = StandardScaler()

# Fit the StandardScaler to the training data
X_scaler = scaler.fit(X_train)

# Scale the training and testing data
scaled_X_train = X_scaler.transform(X_train)
scaled_X_test = X_scaler.transform(X_test)


In [8]:
# Create a OneHotEncoder for the Department column
encoder_department = OneHotEncoder(sparse_output=False,handle_unknown='ignore', drop=None)

# Fit the encoder to the training data
encoder_department.fit(np.array(y_train['Department']).reshape(-1, 1))

# Create two new variables by applying the encoder
# to the training and testing data
y_train_department = encoder_department.transform(np.array(y_train['Department']).reshape(-1, 1))
y_test_department = encoder_department.transform(np.array(y_test['Department']).reshape(-1, 1))

In [9]:
# Create a OneHotEncoder for the Attrition column
encoder_attrition = OneHotEncoder(sparse_output=False,handle_unknown='ignore', drop=None)

# Fit the encoder to the training data
encoder_attrition.fit(np.array(y_train['Attrition']).reshape(-1, 1))

# Create two new variables by applying the encoder
# to the training and testing data
y_train_attrition = encoder_attrition.transform(np.array(y_train['Attrition']).reshape(-1, 1))
y_test_attrition = encoder_attrition.transform(np.array(y_test['Attrition']).reshape(-1, 1))


## Create, Compile, and Train the Model

In [10]:
# Find the number of columns in the X training data
num_features = len(X_train.columns)

# Create the input layer
inputs = layers.Input(shape=(num_features,), name='inputs')

# Create at least two shared layers
shared_layer_1 = layers.Dense(64, activation='relu', name='shared_layer_1')(inputs)
shared_layer_2 = layers.Dense(128, activation='relu', name='shared_layer_2')(shared_layer_1)

In [11]:
# Create a branch for Department # with a hidden layer and an output layer
# Create the hidden layer
department_branch = layers.Dense(32, activation='relu', name='department_branch')(shared_layer_2)

# Create the output layer
department_output = layers.Dense(y_train_department.shape[1], activation='sigmoid', name='department_output')(department_branch)



In [12]:
# Create a branch for Attrition # with a hidden layer and an output layer
# Create the hidden layer
attrition_branch = layers.Dense(32, activation='relu', name='attrition_branch')(shared_layer_2)

# Create the output layer
attrition_output = layers.Dense(y_train_attrition.shape[1], activation='sigmoid', name='attrition_output')(attrition_branch)


In [13]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

In [14]:
# Create the model
model = Model(inputs=inputs, outputs=[department_output, attrition_output])

# Compile the model
model.compile(optimizer='adam',
              loss={
                  'department_output': 'categorical_crossentropy',
                  'attrition_output': 'binary_crossentropy'
              },
              metrics={
                  'department_output': 'accuracy',
                  'attrition_output': 'accuracy'
              })
# Summarize the model
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inputs (InputLayer) │ (None, 11)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_layer_1      │ (None, 64)        │        768 │ inputs[0][0]      │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_layer_2      │ (None, 128)       │      8,320 │ shared_layer_1[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ department_branch   │ (None, 32)        │      4,128 │ shared_layer_2[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attrition_branch    │ (None, 32)        │      4,128 │ shared_layer_2[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ department_output   │ (None, 3)         │         99 │ department_branc… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attrition_output    │ (None, 2)         │         66 │ attrition_branch… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 17,509 (68.39 KB)

 Trainable params: 17,509 (68.39 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
# Train the model (verbose=2 prints one line per epoch to keep the notebook readable)
fit_model = model.fit(scaled_X_train, {'department_output': y_train_department, 'attrition_output': y_train_attrition}, epochs=100, verbose=2)

Epoch 1/100


35/35 - 2s - 71ms/step - attrition_output_accuracy: 0.8403 - attrition_output_loss: 0.5194 - department_output_accuracy: 0.6425 - department_output_loss: 0.8453 - loss: 1.3676


Epoch 2/100


35/35 - 0s - 4ms/step - attrition_output_accuracy: 0.8539 - attrition_output_loss: 0.4194 - department_output_accuracy: 0.6525 - department_output_loss: 0.7741 - loss: 1.1905


Epoch 3/100


...

35/35 - 0s - 4ms/step - attrition_output_accuracy: 1.0000 - attrition_output_loss: 0.0043 - department_output_accuracy: 1.0000 - department_output_loss: 0.0092 - loss: 0.0136


Epoch 99/100


35/35 - 0s - 5ms/step - attrition_output_accuracy: 1.0000 - attrition_output_loss: 0.0041 - department_output_accuracy: 1.0000 - department_output_loss: 0.0082 - loss: 0.0124


Epoch 100/100


35/35 - 0s - 4ms/step - attrition_output_accuracy: 1.0000 - attrition_output_loss: 0.0040 - department_output_accuracy: 1.0000 - department_output_loss: 0.0080 - loss: 0.0120


In [16]:
# Evaluate the model with the testing data.
# return_dict=True so each metric is looked up by name; newer Keras versions
# return a longer list whose order differs from the original version.
test_results = model.evaluate(scaled_X_test, {'department_output': y_test_department, 'attrition_output': y_test_attrition}, return_dict=True)
test_results

 1/12 ━━━━━━━━━━━━━━━━━━━━ 3s 305ms/step - attrition_output_accuracy: 0.7812 - attrition_output_loss: 2.7640 - department_output_accuracy: 0.5938 - department_output_loss: 3.0432 - loss: 5.8071

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - attrition_output_accuracy: 0.7989 - attrition_output_loss: 1.7769 - department_output_accuracy: 0.6005 - department_output_loss: 2.7844 - loss: 4.4841  


{'attrition_output_accuracy': 0.7989130616188049,
 'attrition_output_loss': 1.7769231796264648,
 'department_output_accuracy': 0.6005434989929199,
 'department_output_loss': 2.7843544483184814,
 'loss': 4.484121799468994}

In [17]:
# Print the accuracy for both department and attrition
print(f"Department Accuracy: {test_results['department_output_accuracy']:.3f}")
print(f"Attrition Accuracy: {test_results['attrition_output_accuracy']:.3f}")

Department Accuracy: 0.601
Attrition Accuracy: 0.799


## Sanity check: compare against simple baselines

Added during the 2026 portfolio review. This does not change the model; it puts the accuracy numbers above in context.

- A "majority class" baseline always predicts the most common answer in the training data ("No" for attrition, the largest department for department fit). If the model cannot beat that, its accuracy is not telling us much.
- For attrition, the cases that matter are the employees who actually left, so we also look at how many of them the model caught (recall) and how often a "Yes" flag was right (precision).

In [18]:
from sklearn.metrics import classification_report, confusion_matrix

dept_pred_probs, attr_pred_probs = model.predict(scaled_X_test, verbose=0)

dept_true = encoder_department.categories_[0][y_test_department.argmax(axis=1)]
dept_pred = encoder_department.categories_[0][dept_pred_probs.argmax(axis=1)]
attr_true = encoder_attrition.categories_[0][y_test_attrition.argmax(axis=1)]
attr_pred = encoder_attrition.categories_[0][attr_pred_probs.argmax(axis=1)]

# Majority-class baselines (most common label in the training data, applied to the test set)
dept_majority = y_train['Department'].mode()[0]
attr_majority = y_train['Attrition'].mode()[0]
print(f"Test set size: {len(y_test)} employees")
print(f"Attrition 'Yes' share in test set: {(attr_true == 'Yes').mean():.3f}")
print()
print(f"Department - model accuracy:    {(dept_pred == dept_true).mean():.3f}")
print(f"Department - always '{dept_majority}': {(dept_true == dept_majority).mean():.3f}")
print(f"Attrition  - model accuracy:    {(attr_pred == attr_true).mean():.3f}")
print(f"Attrition  - always '{attr_majority}':      {(attr_true == attr_majority).mean():.3f}")

Test set size: 368 employees
Attrition 'Yes' share in test set: 0.207

Department - model accuracy:    0.601
Department - always 'Research & Development': 0.658
Attrition  - model accuracy:    0.799
Attrition  - always 'No':      0.793


In [19]:
print("Attrition: confusion matrix (rows = actual, columns = predicted)")
print(pd.DataFrame(confusion_matrix(attr_true, attr_pred, labels=['No', 'Yes']),
                   index=['actual No', 'actual Yes'], columns=['pred No', 'pred Yes']))
print()
print(classification_report(attr_true, attr_pred, labels=['No', 'Yes'], zero_division=0))

Attrition: confusion matrix (rows = actual, columns = predicted)
            pred No  pred Yes
actual No       272        20
actual Yes       54        22

              precision    recall  f1-score   support

          No       0.83      0.93      0.88       292
         Yes       0.52      0.29      0.37        76

    accuracy                           0.80       368
   macro avg       0.68      0.61      0.63       368
weighted avg       0.77      0.80      0.78       368



In [20]:
print("Department fit: per-department results")
print(classification_report(dept_true, dept_pred, zero_division=0))

Department fit: per-department results
                        precision    recall  f1-score   support

       Human Resources       0.17      0.06      0.08        18
Research & Development       0.69      0.76      0.72       242
                 Sales       0.39      0.34      0.36       108

              accuracy                           0.60       368
             macro avg       0.41      0.38      0.39       368
          weighted avg       0.57      0.60      0.58       368



# Summary

In the provided space below, briefly answer the following questions.

1. Is accuracy the best metric to use on this data? Why or why not?

2. What activation functions did you choose for your output layers, and why?

3. Can you name a few ways that this model might be improved?


```

YOUR ANSWERS HERE

1. Accuracy might not be the best metric to use on this data, especially if there is class imbalance between the categories in the Department and Attrition columns. Alternative metrics such as precision, recall, or F1-score could provide additional insights into the model's performance.

2. I used the 'sigmoid' activation function for both output layers, as it is suitable for multi-class classification tasks. It allows the model to output a probability distribution over the predicted classes for both the Department and Attrition targets.

3. The model could be improved by adjusting the architecture, such as adding more hidden layers or increasing the number of neurons. Additionally, hyperparameter tuning, using different activation functions, and experimenting with different optimizers could enhance performance. Collecting more data or using techniques to handle class imbalance might also be beneficial. Model can only be as good as its data. 